In [1]:
import os

os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

# A30 GPU — sm_90 architecture
os.environ["TORCH_CUDA_ARCH_LIST"] = "9.0"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import sys
import json
import re
import csv
from pathlib import Path
from typing import Optional

import torch
import transformers
import vllm
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm.auto import tqdm

print("Python executable:", sys.executable)
print("CUDA available:", torch.cuda.is_available())
print("CUDA visible device count:", torch.cuda.device_count())
print("CUDA version:", torch.version.cuda)
print("Torch version:", torch.__version__)
print("transformers:", transformers.__version__)
print("vLLM:", vllm.__version__)

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"cuda:{i} ->", torch.cuda.get_device_name(i))
else:
    raise RuntimeError("CUDA is not available.")

Python executable: /home/folin/private/CSE 151B/151B_SP26_Competition/.venv/bin/python
CUDA available: True
CUDA visible device count: 1
CUDA version: 13.0
Torch version: 2.11.0+cu130
transformers: 5.9.0
vLLM: 0.21.0
cuda:0 -> NVIDIA A30


In [2]:
print("=== HARDWARE CHECK ===")
try:
    a = torch.randn(1000, 1000, device="cuda")
    b = torch.randn(1000, 1000, device="cuda")
    c = torch.matmul(a, b)
    torch.cuda.synchronize()
    print("[SUCCESS] GPU is operational. Output shape:", c.shape)
except Exception as e:
    print("[FAILURE]", e)
print("======================")

=== HARDWARE CHECK ===
[SUCCESS] GPU is operational. Output shape: torch.Size([1000, 1000])


## Configuration

In [3]:
# ── Configuration ──────────────────────────────────────────────────────────────
MODEL_ID   = "Qwen/Qwen3-4B-Thinking-2507"
DATA_PATH  = "data/private.jsonl"
MAX_TOKENS = 16384
K          = 1     # self-consistency samples per question — majority vote picks the answer

print("MODEL_ID:", MODEL_ID)
print("DATA_PATH:", DATA_PATH)
print("MAX_TOKENS:", MAX_TOKENS)
print("K (self-consistency):", K)

MODEL_ID: Qwen/Qwen3-4B-Thinking-2507
DATA_PATH: data/private.jsonl
MAX_TOKENS: 16384
K (self-consistency): 1


In [4]:
# ═══════════════════════════════════════════════════════════════════════════════
# CHUNK / SECTION SELECTOR
#
# The dataset is split into 64 chunks grouped into 8 sections (eighths):
#   s1 = c01–c08  |  s2 = c09–c16  |  s3 = c17–c24  |  s4 = c25–c32
#   s5 = c33–c40  |  s6 = c41–c48  |  s7 = c49–c56  |  s8 = c57–c64
#
# There are THREE independent ways to skip — they all work together:
#
#   1. CHUNKS_TO_RUN  (below) — the master list of all 64 chunks.
#      Comment out individual lines to exclude specific chunks.
#      Best for running a small custom subset.
#
#   2. SECTIONS_TO_SKIP (further below) — skip an entire section (8 chunks) at once.
#      Best for splitting work across multiple runners.
#      Example — you cover s3–s8, someone else has s1–s2:
#          SECTIONS_TO_SKIP = ["s1", "s2"]
#
#   3. CHECKPOINT AUTO-SKIP — if results/sc_k3_private_cXX.csv already exists,
#      that chunk is skipped automatically. Safe to re-run after interruption.
# ═══════════════════════════════════════════════════════════════════════════════

CHUNKS_TO_RUN = [
    # ── Section s1  (first eighth) ─────────────────────────────────────────────
    "c01",  # Chunk  1 of 64
    "c02",  # Chunk  2 of 64
    "c03",  # Chunk  3 of 64
    "c04",  # Chunk  4 of 64
    "c05",  # Chunk  5 of 64
    "c06",  # Chunk  6 of 64
    "c07",  # Chunk  7 of 64
    "c08",  # Chunk  8 of 64
    # ── Section s2 ─────────────────────────────────────────────────────────────
    "c09",  # Chunk  9 of 64
    "c10",  # Chunk 10 of 64
    "c11",  # Chunk 11 of 64
    "c12",  # Chunk 12 of 64
    "c13",  # Chunk 13 of 64
    "c14",  # Chunk 14 of 64
    "c15",  # Chunk 15 of 64
    "c16",  # Chunk 16 of 64
    # ── Section s3 ─────────────────────────────────────────────────────────────
    "c17",  # Chunk 17 of 64
    "c18",  # Chunk 18 of 64
    "c19",  # Chunk 19 of 64
    "c20",  # Chunk 20 of 64
    "c21",  # Chunk 21 of 64
    "c22",  # Chunk 22 of 64
    "c23",  # Chunk 23 of 64
    "c24",  # Chunk 24 of 64
    # ── Section s4 ─────────────────────────────────────────────────────────────
    "c25",  # Chunk 25 of 64
    "c26",  # Chunk 26 of 64
    "c27",  # Chunk 27 of 64
    "c28",  # Chunk 28 of 64
    "c29",  # Chunk 29 of 64
    "c30",  # Chunk 30 of 64
    "c31",  # Chunk 31 of 64
    "c32",  # Chunk 32 of 64
    # ── Section s5 ─────────────────────────────────────────────────────────────
    "c33",  # Chunk 33 of 64
    "c34",  # Chunk 34 of 64
    "c35",  # Chunk 35 of 64
    "c36",  # Chunk 36 of 64
    "c37",  # Chunk 37 of 64
    "c38",  # Chunk 38 of 64
    "c39",  # Chunk 39 of 64
    "c40",  # Chunk 40 of 64
    # ── Section s6 ─────────────────────────────────────────────────────────────
    "c41",  # Chunk 41 of 64
    "c42",  # Chunk 42 of 64
    "c43",  # Chunk 43 of 64
    "c44",  # Chunk 44 of 64
    "c45",  # Chunk 45 of 64
    "c46",  # Chunk 46 of 64
    "c47",  # Chunk 47 of 64
    "c48",  # Chunk 48 of 64
    # ── Section s7 ─────────────────────────────────────────────────────────────
    "c49",  # Chunk 49 of 64
    "c50",  # Chunk 50 of 64
    "c51",  # Chunk 51 of 64
    "c52",  # Chunk 52 of 64
    "c53",  # Chunk 53 of 64
    "c54",  # Chunk 54 of 64
    "c55",  # Chunk 55 of 64
    "c56",  # Chunk 56 of 64
    # ── Section s8  (last eighth) ──────────────────────────────────────────────
    "c57",  # Chunk 57 of 64
    "c58",  # Chunk 58 of 64
    "c59",  # Chunk 59 of 64
    "c60",  # Chunk 60 of 64
    "c61",  # Chunk 61 of 64
    "c62",  # Chunk 62 of 64
    "c63",  # Chunk 63 of 64
    "c64",  # Chunk 64 of 64
]

# ── SECTIONS_TO_SKIP — skip an entire eighth at once ──────────────────────────
#
# Add section names ("s1" through "s8") to exclude all 8 chunks in that section.
# Leave empty to run everything in CHUNKS_TO_RUN (default).
#
# When splitting work across people:
#   - Everyone keeps CHUNKS_TO_RUN as-is (all 64 listed)
#   - Each person lists the OTHER runners' sections here
#
# Example — 4 runners, each covering 2 sections:
#   Runner 1: SECTIONS_TO_SKIP = ["s3","s4","s5","s6","s7","s8"]  → runs s1+s2
#   Runner 2: SECTIONS_TO_SKIP = ["s1","s2","s5","s6","s7","s8"]  → runs s3+s4
#   Runner 3: SECTIONS_TO_SKIP = ["s1","s2","s3","s4","s7","s8"]  → runs s5+s6
#   Runner 4: SECTIONS_TO_SKIP = ["s1","s2","s3","s4","s5","s6"]  → runs s7+s8

SECTIONS_TO_SKIP = ["s1","s2", "s5", "s6", "s7", "s8"]   # ← edit this line; leave CHUNKS_TO_RUN above untouched

# ── CHUNKS_TO_SKIP — skip individual chunks (within sections you are running) ──
#
# Optional fine-grained override on top of SECTIONS_TO_SKIP.
# Example: CHUNKS_TO_SKIP = ["c03", "c07"]

CHUNKS_TO_SKIP = []

# ── Apply filters (do not edit below this line) ───────────────────────────────
_SECTION_MAP = {f"s{i+1}": [f"c{i*8+j+1:02d}" for j in range(8)] for i in range(8)}
_section_excluded = {c for sec in SECTIONS_TO_SKIP for c in _SECTION_MAP.get(sec, [])}
_skip_all = _section_excluded | set(CHUNKS_TO_SKIP)
_chunks_to_run = [c for c in CHUNKS_TO_RUN if c not in _skip_all]

if SECTIONS_TO_SKIP:
    print(f"Sections skipped:     {SECTIONS_TO_SKIP}  ({len(_section_excluded)} chunks)")
if CHUNKS_TO_SKIP:
    print(f"Extra chunks skipped: {CHUNKS_TO_SKIP}")
print(f"Chunks queued to run: {_chunks_to_run}  ({len(_chunks_to_run)} total)")

Sections skipped:     ['s1', 's2', 's5', 's6', 's7', 's8']  (48 chunks)
Chunks queued to run: ['c17', 'c18', 'c19', 'c20', 'c21', 'c22', 'c23', 'c24', 'c25', 'c26', 'c27', 'c28', 'c29', 'c30', 'c31', 'c32']  (16 total)


In [5]:
# ── Load full dataset + build 64-chunk / 8-section slice table ─────────────────
data_path = Path(DATA_PATH)
assert data_path.exists(), f"Cannot find {DATA_PATH}. Run from the competition repo root."

data    = [json.loads(line) for line in open(data_path, encoding="utf-8")]
n_total = len(data)

# 64 chunks derived by subdividing each of 8 equal sections into 8 sub-chunks.
# Section → chunk mapping:
#   s1 = c01–c08  |  s2 = c09–c16  |  s3 = c17–c24  |  s4 = c25–c32
#   s5 = c33–c40  |  s6 = c41–c48  |  s7 = c49–c56  |  s8 = c57–c64
#
# Quarter alignment (for merging with legacy quarter-based outputs):
#   q1 — first 25%  = s1 + s2 = c01–c16
#   q2 — second 25% = s3 + s4 = c17–c32
#   q3 — third 25%  = s5 + s6 = c33–c48
#   q4 — last 25%   = s7 + s8 = c49–c64

s  = n_total // 8
_s = [i * s for i in range(8)] + [n_total]

_slices = {}
for si in range(8):
    ss, se = _s[si], _s[si + 1]
    sub = (se - ss) // 8
    for ci in range(8):
        name = f"c{si * 8 + ci + 1:02d}"
        cs = ss + ci * sub
        ce = ss + (ci + 1) * sub if ci < 7 else se
        _slices[name] = (cs, ce)

print(f"Loaded {n_total} total questions from private.jsonl")
print(f"Built {len(_slices)} chunk slices (~{n_total // 64} questions each)")

Loaded 943 total questions from private.jsonl
Built 64 chunk slices (~14 questions each)


## Prompt Construction

Pure chain-of-thought reasoning — no code generation or execution.
- **MCQ**: model selects the correct letter and wraps it in `\boxed{}`
- **Free-form**: model reasons step-by-step and puts the final answer in `\boxed{}`

In [6]:
# Official Qwen CoT trigger phrase — matches Qwen2.5-Math training distribution exactly
SYSTEM_PROMPT_FREEFORM = """Please Give one answer per [ANS] in the order they appear, all inside a single \\boxed{} separated by commas (e.g. \\boxed{5, 7}).
Put only the value in the box: no \"x =\", no units, no prose. 
If an [ANS] is followed by lettered choices, answer with the letter.""".strip()

SYSTEM_PROMPT_MCQ = """Please Output your final answer as a single capital letter from the given choices, inside \\boxed{} (e.g. \\boxed{C}).
Do not put any formula, number, or option text in the box.""".strip()


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for one competition item."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {str(opt).strip()}" for lbl, opt in zip(labels, options))
        user_prompt = (
            f"Problem:\n{question}\n\nAnswer choices:\n{opts_text}\n\n"
            "Solve the problem and end with the required boxed letter."
        )
        return SYSTEM_PROMPT_MCQ, user_prompt

    user_prompt = (
        f"Problem:\n{question}\n\n"
        "Solve the problem and end with the required boxed answer."
    )
    return SYSTEM_PROMPT_FREEFORM, user_prompt


print("Prompts loaded.")

Prompts loaded.


## Load Model (A30 — bfloat16, 24 GB VRAM)

In [7]:
from transformers.models.qwen2.tokenization_qwen2 import Qwen2Tokenizer

if not hasattr(Qwen2Tokenizer, "all_special_tokens_extended"):
    print("Patching Qwen2Tokenizer.all_special_tokens_extended ...")

    @property
    def all_special_tokens_extended(self):
        return list(self.all_special_tokens)

    Qwen2Tokenizer.all_special_tokens_extended = all_special_tokens_extended
else:
    print("Qwen2Tokenizer already has all_special_tokens_extended.")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    padding_side="left",
    use_fast=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer:", tokenizer.__class__.__name__)

# ── vLLM model — A30 settings ──────────────────────────────────────────────────
# A30: 24 GB VRAM, sm_90, native bfloat16 support
# K=3 means up to 3 sequences in flight per prompt, so max_num_seqs=16 is fine
vllm_model = LLM(
    model=MODEL_ID,
    dtype="bfloat16",
    trust_remote_code=True,
    gpu_memory_utilization=0.92,
    max_model_len=32768,
    max_num_seqs=8,
    max_num_batched_tokens=16384,
    enable_chunked_prefill=True,
    enable_prefix_caching=True,
)

# K=3 self-consistency: generate 3 independent samples per question
# Official Qwen3-Thinking recommended params (no presence_penalty)
sampling_params_sc = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    n=K,
    repetition_penalty=1.0,
)

print(f"Model loaded. Sampling K={K} responses per question.")

Patching Qwen2Tokenizer.all_special_tokens_extended ...


Tokenizer: Qwen2Tokenizer
INFO 05-27 20:06:57 [utils.py:240] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'max_model_len': 32768, 'enable_prefix_caching': True, 'max_num_batched_tokens': 16384, 'max_num_seqs': 8, 'disable_log_stats': True, 'enable_chunked_prefill': True, 'model': 'Qwen/Qwen3-4B-Thinking-2507'}


INFO 05-27 20:06:58 [model.py:568] Resolved architecture: Qwen3ForCausalLM


INFO 05-27 20:06:58 [model.py:1697] Using max model len 32768


INFO 05-27 20:06:58 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=16384.


...

(EngineCore pid=1746) INFO 05-27 20:07:22 [gpu_worker.py:462] Available KV cache memory: 12.81 GiB
(EngineCore pid=1746) INFO 05-27 20:07:22 [gpu_worker.py:477] CUDA graph memory profiling is enabled (default since v0.21.0). The current --gpu-memory-utilization=0.9200 is equivalent to --gpu-memory-utilization=0.9174 without CUDA graph memory profiling. To maintain the same effective KV cache size as before, increase --gpu-memory-utilization to 0.9226. To disable, set VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=0.
(EngineCore pid=1746) INFO 05-27 20:07:22 [kv_cache_utils.py:1710] GPU KV cache size: 93,296 tokens
(EngineCore pid=1746) INFO 05-27 20:07:22 [kv_cache_utils.py:1711] Maximum concurrency for 32,768 tokens per request: 2.85x
(EngineCore pid=1746) INFO 05-27 20:07:23 [kernel_warmup.py:44] Skipping FlashInfer autotune because it is disabled.


Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 5/5 [00:00<00:00, 29.65it/s]
Capturing CUDA graphs (decode, FULL):   0%|          | 0/4 [00:00<?, ?it/s]

Capturing CUDA graphs (decode, FULL): 100%|██████████| 4/4 [00:00<00:00, 32.48it/s]


(EngineCore pid=1746) INFO 05-27 20:07:23 [gpu_model_runner.py:6243] Graph capturing finished in 1 secs, took 0.05 GiB
(EngineCore pid=1746) INFO 05-27 20:07:23 [gpu_worker.py:621] CUDA graph pool memory: 0.05 GiB (actual), 0.06 GiB (estimated), difference: 0.01 GiB (23.1%).
(EngineCore pid=1746) INFO 05-27 20:07:23 [jit_monitor.py:54] Kernel JIT monitor activated — Triton JIT compilations during inference will be logged as warnings.
(EngineCore pid=1746) INFO 05-27 20:07:23 [core.py:299] init engine (profile, create kv cache, warmup model) took 9.41 s (compilation: 4.87 s)


(EngineCore pid=1746) INFO 05-27 20:07:25 [vllm.py:886] Asynchronous scheduling is enabled.
(EngineCore pid=1746) INFO 05-27 20:07:25 [kernel.py:212] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
Model loaded. Sampling K=1 responses per question.


## Generate → Vote → Save (checkpoint loop)\n\nRuns each selected chunk sequentially. Each chunk is saved to its own CSV immediately after generation so progress is never lost if the session dies. Already-saved chunks are skipped automatically on re-run.

In [8]:
def extract_boxed(text: str):
    """Extract last \\boxed{...} content, handling nested braces."""
    marker = r"\boxed{"
    start  = text.rfind(marker)
    if start == -1:
        return None
    i, depth, chars = start + len(marker), 1, []
    while i < len(text):
        ch = text[i]
        if ch == "{":   depth += 1; chars.append(ch)
        elif ch == "}":
            depth -= 1
            if depth == 0: return "".join(chars).strip()
            chars.append(ch)
        else: chars.append(ch)
        i += 1
    return None


def extract_letter(text: str) -> str:
    """Fallback: find a capital letter A-J near the end if boxed extraction fails."""
    if not text:
        return ""
    for pattern in [
        r"answer is ([A-J])", r"answer is: ([A-J])",
        r"answer is \(([A-J])\)", r"Choice ([A-J])", r"Option ([A-J])",
    ]:
        m = re.search(pattern, text, re.IGNORECASE)
        if m:
            return m.group(1).upper()
    m = re.search(r"\b([A-J])\b", text[-50:].upper())
    return m.group(1).upper() if m else ""


print("Extraction helpers loaded.")

Extraction helpers loaded.


In [9]:
def format_chat_prompt(item: dict) -> str:
    system, user = build_prompt(item["question"], item.get("options"))
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=True,
    )

def majority_vote(boxed_answers):
    valid = [b for b in boxed_answers if b is not None]
    if not valid:
        return None, "all_none"
    counts = {}
    for b in valid:
        counts[b] = counts.get(b, 0) + 1
    max_count = max(counts.values())
    winners   = {b for b, c in counts.items() if c == max_count}
    if len(winners) == 1:
        return next(iter(winners)), "majority"
    for b in valid:
        if b in winners:
            return b, "tie_first"


# ── Checkpoint loop ────────────────────────────────────────────────────────────
completed = []
skipped   = []

for chunk in _chunks_to_run:
    out_path = Path(f"results/submission/sc_k1_private_{chunk}.csv")

    if out_path.exists():
        print(f"[SKIP] {chunk} — {out_path} already exists")
        skipped.append(chunk)
        continue

    start, end  = _slices[chunk]
    chunk_data  = data[start:end]

    print(f"\n{'═'*60}")
    print(f"Chunk {chunk}  |  indices [{start}, {end})  |  {len(chunk_data)} questions")
    print(f"{'═'*60}")

    prompts = [format_chat_prompt(item) for item in chunk_data]
    outputs = vllm_model.generate(prompts, sampling_params=sampling_params_sc)

    submission = []
    vote_stats = {"majority": 0, "tie_first": 0, "all_none": 0}

    for item, out in zip(chunk_data, outputs):
        texts = [o.text.strip() for o in out.outputs]
        boxed = [extract_boxed(t) for t in texts]
        voted, status = majority_vote(boxed)
        vote_stats[status] = vote_stats.get(status, 0) + 1
        rep_idx  = next((i for i, b in enumerate(boxed) if b == voted), 0)
        submission.append({"id": item["id"], "response": texts[rep_idx]})

    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["id", "response"])
        writer.writeheader()
        writer.writerows(submission)

    print(f"[SAVED] {len(submission)} rows → {out_path}")
    print(f"  Vote: majority={vote_stats['majority']}  "
          f"tie={vote_stats['tie_first']}  all_none={vote_stats['all_none']}")
    completed.append(chunk)

print(f"\n{'═'*60}")
print(f"Done.  Completed: {completed}")
print(f"       Skipped (already existed): {skipped}")
print(f"{'═'*60}")


════════════════════════════════════════════════════════════
Chunk c17  |  indices [234, 248)  |  14 questions
════════════════════════════════════════════════════════════


Rendering prompts:   0%|          | 0/14 [00:00<?, ?it/s]

(EngineCore pid=1746) WARNING 05-27 20:07:25 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _compute_slot_mapping_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts:   0%|          | 0/14 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

[SAVED] 14 rows → results/submission/sc_k1_private_c17.csv
  Vote: majority=12  tie=0  all_none=2

════════════════════════════════════════════════════════════
Chunk c18  |  indices [248, 262)  |  14 questions
════════════════════════════════════════════════════════════


...

Processed prompts:   0%|          | 0/14 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

[SAVED] 14 rows → results/submission/sc_k1_private_c31.csv
  Vote: majority=10  tie=0  all_none=4

════════════════════════════════════════════════════════════
Chunk c32  |  indices [449, 468)  |  19 questions
════════════════════════════════════════════════════════════


Rendering prompts:   0%|          | 0/19 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/19 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

[SAVED] 19 rows → results/submission/sc_k1_private_c32.csv
  Vote: majority=14  tie=0  all_none=5

════════════════════════════════════════════════════════════
Done.  Completed: ['c17', 'c18', 'c19', 'c20', 'c21', 'c22', 'c23', 'c24', 'c25', 'c26', 'c27', 'c28', 'c29', 'c30', 'c31', 'c32']
       Skipped (already existed): []
════════════════════════════════════════════════════════════
